<div dir="rtl" style="text-align:right">

# کارگاه یادگیری ماشین با scikit-learn — جلسه ۲۲

**درس:** مقدمهای بر هوش مصنوعی و کاربرد آن در مهندسی پزشکی

در این کارگاه چرخه‌ی کامل یادگیری ماشین (جلسه ۲۱) را با کد واقعی اجرا می‌کنیم:
داده → تفکیک آموزش/آزمون → آموزش → ارزیابی → تفسیر.

> [!NOTE]
> مثال صرفاً آموزشی و غیرتشخیصی است؛ دیتاست `breast_cancer` برای تمرین برنامه‌نویسی است، نه تصمیم بالینی.

</div>

<div dir="rtl" style="text-align:right">

## گام ۰: وارد کردن کتابخانه‌ها
همه‌ی seedها را روی ۴۲ ثابت می‌کنیم تا نتایج شما با نتایج کلاس یکسان بماند.

</div>

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

SEED = 42

<div dir="rtl" style="text-align:right">

## گام ۱: بارگذاری داده
۵۶۹ نمونه، ۳۰ ویژگی عددی از تصویربرداری سلول‌ها؛ برچسب: ۰ = بدخیم، ۱ = خوش‌خیم.

</div>

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
print("shape:", X.shape)
print("کلاس‌ها:", dict(zip(data.target_names.tolist(), np.bincount(y).tolist())))
print("سه ویژگی اول:", data.feature_names[:3])

<div dir="rtl" style="text-align:right">

## گام ۲: تفکیک آموزش/آزمون
چرا `stratify=y`؟ چون نسبت بدخیم/خوش‌خیم در هر دو قسمت یکسان بماند (داده کمی نامتوازن است).

</div>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print("train:", X_train.shape, " test:", X_test.shape)

<div dir="rtl" style="text-align:right">

## گام ۳: آموزش یک مدل ساده — رگرسیون لجستیک
اولین مدل؛ ساده و قابل تفسیر.

</div>

In [ ]:
log_reg = LogisticRegression(max_iter=5000, random_state=SEED)
log_reg.fit(X_train, y_train)
y_pred = log_reg.predict(X_test)
print("accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("f1:", round(f1_score(y_test, y_pred), 3))

<div dir="rtl" style="text-align:right">

## گام ۴: ماتریس درهم‌ریختگی
ترتیب sklearn: سطر = واقعی، ستون = پیش‌بینی؛ کلاس ۰ (بدخیم) اول است.

</div>

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)
# ردیف/ستون اول = کلاس 0 (بدخیم)؛ حساسیت نسبت به بدخیم = cm[0,0] / (cm[0,0] + cm[0,1])
sens_malignant = cm[0, 0] / (cm[0, 0] + cm[0, 1])
print("sensitivity (کلاس بدخیم):", round(sens_malignant, 3))

<div dir="rtl" style="text-align:right">

## گام ۵: مقایسه با درخت تصمیم و جنگل تصادفی
درخت را عمداً بدون محدودیت عمق می‌سازیم تا بیش‌برازش را ببینیم.

</div>

In [ ]:
tree = DecisionTreeClassifier(random_state=SEED)
tree.fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf.fit(X_train, y_train)

print("درخت عمیق:  train =", round(tree.score(X_train, y_train), 3),
      " test =", round(tree.score(X_test, y_test), 3))
print("جنگل:        train =", round(rf.score(X_train, y_train), 3),
      " test =", round(rf.score(X_test, y_test), 3))

<div dir="rtl" style="text-align:right">

## گام ۶: درمان بیش‌برازش با محدود کردن عمق

</div>

In [ ]:
for depth in [2, 3, 5, 10]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    t.fit(X_train, y_train)
    print(f"max_depth={depth:>2}: train={t.score(X_train, y_train):.3f}"
          f"  test={t.score(X_test, y_test):.3f}")

<div dir="rtl" style="text-align:right">

## گام ۷: کدام ویژگی‌ها مهم بودند؟
اهمیت ویژگی (Feature Importance) جنگل تصادفی.

</div>

In [ ]:
importances = rf.feature_importances_
order = np.argsort(importances)[::-1][:5]
for i in order:
    print(f"{data.feature_names[i]:<25} {importances[i]:.3f}")

<div dir="rtl" style="text-align:right">

## ✍️ تمرین کارگاه
۱. `test_size` را به ۰٫۵ تغییر دهید؛ Accuracy چه تغییری کرد؟ چرا؟
۲. `random_state` را عوض کنید؛ آیا نتیجه جابه‌جا می‌شود؟ از این نتیجه چه نتیجه‌ای درباره‌ی «یک عدد Accuracy» می‌گیرید؟
۳. `max_depth` بهینه را پیدا کنید و توضیح دهید چرا درخت عمیق‌تر بدتر شد.

راهنما: کد پایتون معادل این نوت‌بوک در `code/session_22_sklearn_workshop.py` است.

</div>